# OG-001: Private source transport and environment validation

This runner validates a pinned history-free source snapshot and the Kaggle T4 dependency overlay. It does not execute a scientific experiment. Original source identity, snapshot identity, and notebook commit are recorded separately.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import hashlib
import importlib.util
import json
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import traceback
import uuid
from datetime import datetime, timezone

SOURCE_ORIGIN_SHA = "6cb7933dd4eb4d3753bc3535aae6388c6a1d2236"
SOURCE_TREE_SHA = "23e501ef34cb33e6b0840cf84c0e4e532449063b"
SOURCE_MANIFEST_SHA256 = "32336cd78b174ac5471dad52fc57660d436505cba46d7542f7fc1e8ec95d4eea"
DATASET_ID = "muhammadii/fq-ssiuu-source-6cb7933d"
# Staging fills this parameter with the commit containing this notebook.
ORCHESTRATION_COMMIT = "82e8736fbdb22b7e807cfaa7d2ace3791afd494a"
RUN_INTEGRATION_TESTS = True


def sha256(path):
    with open(path, "rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def write_json(path, data):
    with tempfile.NamedTemporaryFile(mode="w", dir=path.parent, delete=False) as handle:
        json.dump(data, handle, indent=2)
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(handle.name, path)


def checked_git(cwd, *args):
    result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"git {args[0]} failed, exit {result.returncode}")
    return result.stdout.strip()


def run_validation(input_root=Path("/kaggle/input"), working=Path("/kaggle/working")):
    working.mkdir(parents=True, exist_ok=True)
    repo = working / "fq-ssiuu"
    output = working / "og001_output"
    summary = {
        "schema_version": "2.0", "orchestration_run_id": str(uuid.uuid4()),
        "start_utc": datetime.now(timezone.utc).isoformat(),
        "source_origin_sha": SOURCE_ORIGIN_SHA, "source_tree_sha": SOURCE_TREE_SHA,
        "source_manifest_sha256": SOURCE_MANIFEST_SHA256,
        "orchestration_git_commit": ORCHESTRATION_COMMIT,
        "dataset_id": DATASET_ID, "source_validation_status": "NOT_REACHED",
        "setup_returncode": None, "execution_status": "FAIL",
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "scientific_experiment_executed": False,
    }
    phase = "source_validation"
    owned_repo = False
    try:
        if ORCHESTRATION_COMMIT == "STAGING_REQUIRED":
            raise RuntimeError("Notebook staging provenance has not been supplied")
        candidates = [p for p in input_root.rglob("SOURCE_MANIFEST.json")
                      if sha256(p) == SOURCE_MANIFEST_SHA256]
        if len(candidates) != 1:
            raise RuntimeError("Expected exactly one attached source manifest with the pinned SHA-256")
        manifest_path = candidates[0]
        manifest = json.loads(manifest_path.read_text())
        if (manifest.get("source_origin_sha") != SOURCE_ORIGIN_SHA
                or manifest.get("source_tree_sha") != SOURCE_TREE_SHA
                or manifest.get("snapshot_ref") != "kaggle-snapshot"
                or manifest.get("bundle_filename") != "fq-ssiuu-source.bundle"):
            raise RuntimeError("Source manifest provenance mismatch")
        bundle = manifest_path.parent / "fq-ssiuu-source.bundle"
        if (sha256(bundle) != manifest["bundle_sha256"]
                or bundle.stat().st_size != manifest["bundle_size_bytes"]):
            raise RuntimeError("Source bundle SHA-256 or size mismatch")
        if repo.exists() or output.exists():
            raise RuntimeError("Source checkout or setup output already exists")
        checked_git(working, "clone", "--no-checkout", "-b", "kaggle-snapshot", str(bundle), str(repo))
        owned_repo = True
        head = checked_git(repo, "rev-parse", "HEAD")
        tree = checked_git(repo, "rev-parse", "HEAD^{tree}")
        if head != manifest["snapshot_commit_sha"] or tree != SOURCE_TREE_SHA:
            raise RuntimeError("Cloned snapshot HEAD or tree mismatch")
        if checked_git(repo, "rev-list", "--count", "--all") != "1":
            raise RuntimeError("Source snapshot contains history")
        if len(checked_git(repo, "rev-list", "--parents", "-n", "1", "HEAD").split()) != 1:
            raise RuntimeError("Source snapshot has a parent")
        checked_git(repo, "checkout", "kaggle-snapshot")
        if checked_git(repo, "status", "--porcelain"):
            raise RuntimeError("Source snapshot is dirty")
        # Only load infrastructure code after its entire source tree is verified.
        spec = importlib.util.spec_from_file_location("source_transport", repo / "scripts/package_kaggle_source.py")
        transport = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(transport)
        transport.verify_bundle(manifest_path.parent, SOURCE_ORIGIN_SHA, SOURCE_MANIFEST_SHA256)
        summary.update({"source_validation_status": "PASS", "snapshot_commit_sha": head,
                        "bundle_sha256": manifest["bundle_sha256"], "history_commit_count": 1})
        write_json(working / "SOURCE_MANIFEST.json", manifest)
        phase = "setup"
        setup = subprocess.run([sys.executable, str(repo / "scripts/kaggle_setup.py"),
                                "--apply", "--output-dir", str(output)],
                               cwd=repo, capture_output=True, text=True)
        summary["setup_returncode"] = setup.returncode
        (working / "kaggle_runner_stdout.log").write_text(setup.stdout)
        (working / "kaggle_runner_stderr.log").write_text(setup.stderr)
        if setup.returncode:
            raise RuntimeError(f"Environment setup failed with exit {setup.returncode}; see setup logs")
        phase = "evidence_validation"
        # Import only from the source tree already verified above.
        if not (output / "setup_status.json").is_file():
            raise RuntimeError("Missing setup evidence: setup_status.json")
        sys.path.insert(0, str(repo))
        from scripts.validate_og001_artifacts import validate_setup_artifacts
        verdict = validate_setup_artifacts(output, head, repo / "requirements/kaggle-native-baseline.json")
        write_json(working / "og001_acceptance.json", verdict)
        summary["setup_run_id"] = verdict["run_id"]
        summary["pip_consistency_state"] = verdict["pip_consistency_state"]
        summary["og001_status"] = "PASS"
        if RUN_INTEGRATION_TESTS:
            phase = "integration_tests"
            integration_env = dict(os.environ, HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
                                   TRANSFORMERS_OFFLINE="1", PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
            notebook_check = subprocess.run(
                [sys.executable, str(repo / "scripts/check_notebooks.py"),
                 "--output", str(working / "notebook_validation.json")], cwd=repo, capture_output=True)
            (working / "notebook_validation_stdout.log").write_bytes(notebook_check.stdout)
            (working / "notebook_validation_stderr.log").write_bytes(notebook_check.stderr)
            summary["notebook_validation_returncode"] = notebook_check.returncode
            integration = subprocess.run(
                [sys.executable, "-m", "pytest", "tests", "-m", "not requires_model", "-ra",
                 "--junitxml=" + str(working / "integration-tests.xml")],
                cwd=repo, capture_output=True, env=integration_env)
            (working / "integration_stdout.log").write_bytes(integration.stdout)
            (working / "integration_stderr.log").write_bytes(integration.stderr)
            summary["integration_returncode"] = integration.returncode
            if integration.returncode or notebook_check.returncode:
                raise RuntimeError("Non-model integration suite failed; OG-001 acceptance retained separately")
        summary["execution_status"] = "PASS"
    except Exception as exc:
        summary["failure_phase"] = phase
        summary["first_failure"] = f"{type(exc).__name__}: {exc}"
        (working / "orchestration_failure.log").write_text(traceback.format_exc())
    finally:
        # The generated checkout is disposable; all run evidence is outside it.
        if owned_repo:
            try:
                shutil.rmtree(repo)
            except OSError as exc:
                summary["cleanup_error"] = type(exc).__name__
        summary["end_utc"] = datetime.now(timezone.utc).isoformat()
        summary["setup_status_file_exists"] = (output / "setup_status.json").is_file()
        write_json(working / "kernel_summary.json", summary)
        print(json.dumps(summary, indent=2))
    if summary["execution_status"] != "PASS":
        raise RuntimeError(summary.get("first_failure", "Validation failed"))
    return summary


if __name__ == "__main__":
    run_validation()
